In [1]:
""" Used when checked out from Git """
import sys
import os
def add_local():
    d = os.getcwd()
    assert d[-9:] == "notebooks", "This code assume that the notebook is run out of 'notebooks'. Remove this code if not needed"
    sys.path.insert( 0, d[:-10] )
    print(f"Added '{d[:-10]}' to the import path")
add_local()


Added '/home/hans/python/packages/cdxcore' to the import path


In [2]:
from cdxcore.version import version

class A(object):
    def __init__(self, x=2):
        self.x = x
    @version(version="0.4.1")
    def h(self, y):
        return self.x*y

@version(version="0.3.0")
def h(x,y):
    return x+y

@version(version="0.0.2", dependencies=[h])
def f(x,y):
    return h(y,x)

@version(version="0.0.1", dependencies=["f", A.h])
def g(x,z):
    a = A()
    return f(x*2,z)+a.h(z)

g(1,2)
print("version", g.version.input)                # -> version 0.0.1
print("full version", g.version.full )           # -> full version 0.0.1 { f: 0.0.2 { h: 0.3.0 }, A.h: 0.4.1 }
print("full version ID",g.version.unique_id48 )  # -> full version ID 0.0.1 { f: 0.0.2 { h: 0.3.0 }, A.h: 0.4.1 }
print("full version ID",g.version.unique_id60 )  # -> full version ID 0.0.1 { f: 0.0.2 { h: 0.3.0 }, A.h: 0.4.1 }
print("depedencies",g.version.dependencies )     # -> depedencies ('0.0.1', {'f': ('0.0.2', {'h': '0.3.0'}), 'A.h': '0.4.1'})

version 0.0.1
full version 0.0.1 { A.h: 0.4.1, f: 0.0.2 { h: 0.3.0 } }
full version ID 0.0.1 { A.h: 0.4.1, f: 0.0.2 { h: 0.3.0 } }
full version ID 0.0.1 { A.h: 0.4.1, f: 0.0.2 { h: 0.3.0 } }
depedencies ('0.0.1', {'f': ('0.0.2', {'h': '0.3.0'}), 'A.h': '0.4.1'})


In [3]:
@version("0.1")
class A(object):
    @version("0.2") # automatically depends on A
    def f(self, x):
        return x
    @version("0.3", auto_class=False ) # does not depend on A
    def g(self, x):
        return x
    
@version("0.4") # automatically depends on A
class B(A):
    pass

@version("0.4", auto_class=False ) # does not depend on A
class C(A):
    pass

b = B()
c = C()
print( "B", b.version.full )
print( "C", c.version.full )


B 0.4 { A: 0.1 }
C 0.4


In [4]:
@version("0.0.2", dependencies=[f])
def g(x):
    print(g.version.full) # -> 0.0.2 { f: 0.0.01 }
    return f(x,x)
g(1)

0.0.2 { f: 0.0.2 { h: 0.3.0 } }


2

In [5]:
@version("0.0.4", dependencies=['f'])
def r(x):
    return x

print( r.version.full )    # -> 0.0.4 { f: 0.0.01 }


0.0.4 { f: 0.0.2 { h: 0.3.0 } }


In [6]:
@version("0.0.3", dependencies=[f] )
class A(object):
    def h(self, x):
        return f(x)

print( A.version.input )  # -> 0.0.3
print( A.version.full )   # -> 0.0.3 { f: 0.0.01 }

a = A()
print( a.version.input )  # -> 0.0.3
print( a.version.full )   # -> 0.0.3 { f: 0.0.01 }


0.0.3
0.0.3 { f: 0.0.2 { h: 0.3.0 } }
0.0.3
0.0.3 { f: 0.0.2 { h: 0.3.0 } }


In [7]:
@version("0.0.1")
class A(object):
    pass

@version("0.0.2")
class B(A):
    pass

print( A.version.full )   # -> 0.0.1
print( B.version.full )   # -> 0.0.2 { A: 0.0.1 }

0.0.1
0.0.2 { A: 0.0.1 }


In [3]:
print2 = version("0.1")(print)

In [6]:
print2.version

0.1

In [34]:
class A(object):
    def __new__(cls, *args, **kwargs ):
        return object.__new__(cls)
    def __init__(self, x=1):
        self.x = x

import inspect

s = inspect.signature( A.__new__ )
print(s)
s = inspect.signature( object.__new__ )
for p, pd in s.parameters.items():
    print(p, pd.kind == pd.VAR_POSITIONAL, pd.kind == pd.VAR_KEYWORD )
print(s)
args = [1]
kwargs = {}
a = s.bind(*args,**kwargs)
a.apply_defaults()
a = a.arguments

print(a)

    
        

import types
A.__new__ = version("0.1")(A.__new__)
 
a = A()
A.__new__.__name__
a = A(x=1)
A.__new__.__name__
a = A(1)
A.__new__.__name__



(cls, *args, **kwargs)
args True False
kwargs False True
(*args, **kwargs)
{'args': (1,), 'kwargs': {}}


'__new__'

In [11]:
_str_ = version("0.1")(a.__str__)


AttributeError: Failed to assign 'version' element to type <class 'method-wrapper'>: 'method-wrapper' object has no attribute 'version' and no __dict__ for setting new attributes